# Acervo que Fala — Notebook 03: RAG e a redação v3

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

O Notebook 02 revelou que os erros restantes do pipeline nascem na **redação** (uso errado de "close", artefatos vazando para o alt-text, cores vagas). Este notebook ataca isso com duas armas:

1. **RAG** (*Retrieval-Augmented Generation* — geração aumentada por recuperação): em vez de colocar todas as regras e diretrizes no prompt, o sistema mantém uma **base de conhecimento** (regras gerais, diretrizes por categoria de objeto e um glossário dos termos do catálogo) e **recupera só os trechos relevantes** para cada objeto. Uma tanga de miçangas recebe as diretrizes de miçangaria; um pote, as de cerâmica.
2. **Redação v3**: as quatro falhas do Notebook 02 viram regras explícitas.

Detalhe de método: este notebook **reaproveita as observações visuais do Notebook 02** (salvas no Drive) — não roda a etapa de visão de novo. É a vantagem de separar as etapas: cada uma pode ser refeita ou reaproveitada sozinha.

*Metodologia: projeto construído por um designer com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo: **~12 min**. O Colab pedirá autorização do Drive logo no início (a base de conhecimento e as observações vêm de lá).

## Etapa 1 — Instalar as ferramentas (~2 min)

Além das ferramentas dos notebooks anteriores (com a Pillow travada, regra da casa), entra a **sentence-transformers** — biblioteca que executa modelos de *embeddings*, explicados na Etapa 3.

In [ ]:
import PIL
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers pillow=={PIL.__version__}
print(f"ferramentas instaladas ✓ (Pillow mantida em {PIL.__version__})")

In [ ]:
# Checagem rápida do ambiente (mesma dos notebooks anteriores)
import torch, transformers
from PIL import ImageDraw
from torchvision.io import decode_image
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")

## Etapa 2 — Carregar a base de conhecimento e as observações

Dois arquivos vêm do Drive do projeto:

- **`dados/rubrica.json`** — a base de conhecimento do RAG, escrita para este projeto e versionada no repositório: 8 regras gerais, 11 diretrizes por categoria e 10 termos de glossário. O glossário foi construído a partir dos termos que aparecem no próprio catálogo do museu (acordelado, trançado sarjado, gregas...), traduzidos para linguagem simples.
- **`resultados/02_nivel1_smoke_test.json`** — as observações visuais dos 5 objetos, geradas no Notebook 02, reaproveitadas aqui.

In [ ]:
import json, os
from google.colab import drive

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"

with open(f"{PROJETO}/dados/rubrica.json", encoding="utf-8") as f:
    rubrica = json.load(f)
trechos = rubrica["trechos"]

with open(f"{PROJETO}/resultados/02_nivel1_smoke_test.json", encoding="utf-8") as f:
    resultado02 = json.load(f)
objetos = resultado02["itens"]

print(f"rubrica: {len(trechos)} trechos ({rubrica['versao']})")
print(f"observações reaproveitadas: {len(objetos)} objetos ✓")

## Etapa 3 — Embeddings: como o sistema "encontra" o trecho certo

Um **embedding** transforma um texto numa lista de números que representa o seu significado — textos que falam de coisas parecidas viram números próximos. É o que permite buscar por *sentido*, não por palavra exata: "tecido de contas coloridas" encontra a diretriz de "miçangaria" mesmo sem usar a palavra miçanga.

O modelo usado é o **Qwen3-Embedding-0.6B** (aberto, pequeno — roda ao lado do modelo principal sem pesar na GPU). Cada trecho da rubrica vira um vetor, calculado uma única vez.

*Nota de engenharia:* com 30 trechos, a busca é feita por comparação direta de similaridade — transparente e auditável. Um banco vetorial (ChromaDB) entra quando a base crescer, no site do projeto.

In [ ]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
vetores = embedder.encode([t["texto"] for t in trechos], convert_to_tensor=True)

def recuperar(consulta, k=3, excluir_geral=True):
    """Devolve os k trechos mais próximos da consulta (as regras gerais
    são fixas no prompt, então a busca foca em categoria + glossário)."""
    v = embedder.encode(consulta, convert_to_tensor=True)
    scores = util.cos_sim(v, vetores)[0]
    ordem = scores.argsort(descending=True)
    achados = []
    for i in ordem.tolist():
        if excluir_geral and trechos[i]["categoria"] == "geral":
            continue
        achados.append((float(scores[i]), trechos[i]))
        if len(achados) == k:
            break
    return achados

print(f"{len(trechos)} trechos indexados ✓")

## Etapa 4 — Teste da recuperação (a verificação desta etapa)

Antes de usar o RAG na redação, ele precisa provar que funciona: uma consulta sobre **tanga de miçangas** tem que trazer as diretrizes de miçangaria/têxtil — e **não** as de cerâmica. E vice-versa.

In [ ]:
testes = {
    "tanga de miçangas branca com grafismos pretos, gregas e franjas nas bordas": "miçanga",
    "pote de cerâmica pintada com grafismos geométricos em vermelho e preto": "ceramica",
}
tudo_ok = True
for consulta, esperado in testes.items():
    achados = recuperar(consulta)
    ids = [t["id"] for _, t in achados]
    ok = any(esperado in i for i in ids)
    tudo_ok = tudo_ok and ok
    print(f"CONSULTA: {consulta[:60]}...")
    for score, t in achados:
        print(f"  {score:.3f}  [{t['id']}] {t['texto'][:80]}...")
    print(f"  → esperava trecho '{esperado}-*': {'✓' if ok else '✗ FALHOU'}\n")
print("recuperação verificada ✓" if tudo_ok else "⚠ recuperação falhou — avisar o Claude")

## Etapa 5 — Redação v3: regras endurecidas + diretrizes recuperadas (~6 min)

O prompt de redação agora tem duas partes:

- **Regras fixas** (sempre presentes): as 4 falhas do Notebook 02 viram proibições explícitas — *"inteiro"/"detalhe" como termos exclusivos e nada de "close" para objeto inteiro; artefatos de estúdio NUNCA no alt-text; cores nomeadas, sem "tons terrosos"; incerteza preservada*.
- **Diretrizes recuperadas** (mudam por objeto): os 3 trechos que o RAG achou para a categoria e o conteúdo daquele objeto.

O modelo de redação é o mesmo Qwen3-VL, em modo texto (sem receber a imagem), como no Notebook 02.

In [ ]:
import re
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"
quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = Qwen3VLForConditionalGeneration.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)

def gerar_texto(prompt, max_tokens=200):
    conversa = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

PROMPT_REDACAO_V3 = """Você escreve alt-text de acessibilidade para o acervo digital de um museu, lido por pessoas cegas via leitor de tela.

OBSERVAÇÃO VISUAL DA FOTOGRAFIA (única fonte do que é visível):
{observacao}

DADOS DO REGISTRO DO MUSEU (use SOMENTE nome e povo):
Nome do item: {nome} | Povo: {povo}

DIRETRIZES PARA ESTE TIPO DE OBJETO (recuperadas da base do projeto):
{diretrizes}

REGRAS FIXAS:
1. Uma frase, no máximo 30 palavras, começando pelo objeto e povo.
2. Linguagem simples, sem jargão de catalogação.
3. Enquadramento com termos exclusivos: se a observação diz que o objeto aparece INTEIRO, descreva-o normalmente e NUNCA use a palavra 'close'; se a observação diz que é só um DETALHE, comece o alt-text com 'Detalhe de...'.
4. Artefatos de estúdio ou inventário (etiqueta, numeração, cartela/paleta de cores, régua, suporte) NUNCA aparecem no alt-text.
5. Nomeie as cores como estão na observação (vermelho, preto, amarelo...). Proibido 'colorido', 'tons variados', 'tons terrosos'.
6. Preserve a incerteza da observação: palpite ('provavelmente vime') vira termo genérico ('fibra vegetal') — nunca vira afirmação.
7. Responda APENAS com JSON: {{"alt_text": "..."}}"""

for obj in objetos:
    consulta = f"{obj['titulo']}. {obj['observacao'][:300]}"
    achados = recuperar(consulta)
    obj["diretrizes_usadas"] = [t["id"] for _, t in achados]
    diretrizes = "\n".join(f"- {t['texto']}" for _, t in achados)
    prompt = PROMPT_REDACAO_V3.format(
        observacao=obj["observacao"],
        nome=obj["titulo"], povo="",
        diretrizes=diretrizes,
    )
    resposta = gerar_texto(prompt)
    try:
        obj["alt_text_v3"] = extrair_json(resposta)["alt_text"]
        obj["json_valido_v3"] = True
    except Exception:
        obj["alt_text_v3"] = resposta
        obj["json_valido_v3"] = False
    print(f"— {obj['titulo']} (diretrizes: {obj['diretrizes_usadas']}):\n  {obj['alt_text_v3']}\n")

## Etapa 6 — Antes × depois: a redação v2 (Notebook 02) contra a v3

A tabela compara os alt-texts do Notebook 02 (redação sem RAG) com os novos, e roda as checagens automáticas das 4 falhas: uso indevido de "close", vazamento de artefatos, limite de 30 palavras e JSON válido.

In [ ]:
TERMOS_ARTEFATO = ["cartela", "paleta", "numeração", "marcação", "etiqueta", "régua", "suporte"]
OBJETOS_DETALHE = {63283}  # só o Abano é foto-detalhe entre os 5

def checar(obj, alt):
    problemas = []
    eh_detalhe = obj["id"] in OBJETOS_DETALHE
    if not eh_detalhe and re.search(r"close", alt, re.IGNORECASE):
        problemas.append("'close' em objeto inteiro")
    if eh_detalhe and not re.match(r"detalhe", alt.strip(), re.IGNORECASE):
        problemas.append("detalhe sem aviso inicial")
    for termo in TERMOS_ARTEFATO:
        if termo in alt.lower():
            problemas.append(f"artefato no alt ('{termo}')")
    if len(alt.split()) > 30:
        problemas.append(f"{len(alt.split())} palavras (>30)")
    return problemas

print("=" * 100)
aprovados = 0
for obj in objetos:
    p_v2 = checar(obj, obj["alt_text"])
    p_v3 = checar(obj, obj["alt_text_v3"])
    if not p_v3:
        aprovados += 1
    print(f"\n### {obj['titulo']} ({obj['id']})")
    print(f"V2 (nb02): {obj['alt_text']}")
    print(f"   problemas v2: {p_v2 or 'nenhum'}")
    print(f"V3 (RAG):  {obj['alt_text_v3']}")
    print(f"   problemas v3: {p_v3 or 'nenhum ✓'}")
    print(f"GABARITO:  {obj['gabarito']}")
print("\n" + "=" * 100)
print(f"Resumo v3: {aprovados}/5 sem problemas | {sum(1 for o in objetos if o['json_valido_v3'])}/5 JSONs válidos")

In [ ]:
# Salvar no Drive (mesmo padrão dos demais notebooks)
resultado = {
    "notebook": "03_rag_redacao_v1",
    "modelo": MODELO,
    "embedding": "Qwen/Qwen3-Embedding-0.6B",
    "rubrica_versao": rubrica["versao"],
    "prompt_redacao_v3": PROMPT_REDACAO_V3,
    "itens": [
        {"id": o["id"], "titulo": o["titulo"], "observacao": o["observacao"],
         "alt_v2": o["alt_text"], "alt_v3": o["alt_text_v3"],
         "json_valido_v3": o["json_valido_v3"],
         "diretrizes_usadas": o["diretrizes_usadas"], "gabarito": o["gabarito"]}
        for o in objetos
    ],
}
destino = f"{PROJETO}/resultados/03_rag_redacao.json"
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")

---

## Fim — o que fazer agora

Avise o Claude que o Notebook 03 terminou — ele busca o resultado no Drive e analisa o antes × depois (a pergunta central: as 4 falhas do Notebook 02 sumiram na v3?).

**O que este notebook provou:** a recuperação por significado funciona (diretriz certa para cada categoria) e a redação passou a ser guiada por regras fixas + diretrizes recuperadas. **O que ainda não provou:** o nível 2 (descrição do objeto com o registro completo), as flags de divergência e a escala dos 40 casos — assunto do Notebook 04 (E7).